# 03 — Net PnL per Trade

**V4**: `net_pnl = fill_size × |spread| - fill_size × (fee_bps + slippage_bps) / 10000`  
**V7**: all profit claims cite `net_pnl > 0` after deduction, not gross spread.

If fills table is empty (engine not yet in live-order mode), shows theoretical edge per emitted signal instead.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src import config

cfg = config.load()
fee_bps      = cfg['fee_bps']
slippage_bps = cfg['slippage_bps']
cost_rate    = (fee_bps + slippage_bps) / 10_000
print(f'fee_bps={fee_bps}, slippage_bps={slippage_bps}, cost_rate={cost_rate}')

In [ ]:
joined = pd.read_parquet('../data/raw/joined.parquet')
filled = joined.dropna(subset=['fill_price', 'fill_size']).copy()

if len(filled) == 0:
    print('No fills yet — engine not in live-order mode.')
    print('Showing theoretical edge per emitted signal (V7: gross only, no fill size).')

    emitted = joined[joined['decision'] == 'emitted'].copy()
    emitted['spread']         = pd.to_numeric(emitted['spread'])
    emitted['expected_edge']  = pd.to_numeric(emitted['expected_edge'])
    emitted['gross_edge']     = emitted['spread'].abs()
    emitted['net_edge']       = emitted['gross_edge'] - cost_rate
    profitable = emitted['net_edge'] > 0

    print(f'Emitted signals       : {len(emitted)}')
    print(f'Net edge > 0 (V7)     : {profitable.sum()} / {len(emitted)} ({profitable.mean():.1%})')
    print(f'Mean gross edge       : {emitted["gross_edge"].mean():.4f}')
    print(f'Mean net edge         : {emitted["net_edge"].mean():.4f}')
    print(f'Max net edge          : {emitted["net_edge"].max():.4f}')

    emitted.to_parquet('../data/raw/filled_pnl.parquet', index=False)
    print('Saved data/raw/filled_pnl.parquet (signal-level, no fills)')
else:
    filled['fill_size'] = pd.to_numeric(filled['fill_size'])
    filled['spread']    = pd.to_numeric(filled['spread'])

    # V4 formula
    filled['gross_pnl'] = filled['fill_size'] * filled['spread'].abs()
    filled['cost']      = filled['fill_size'] * cost_rate
    filled['net_pnl']   = filled['gross_pnl'] - filled['cost']
    profitable = filled['net_pnl'] > 0

    print(f'Filled trades         : {len(filled)}')
    print(f'Profitable (net, V7)  : {profitable.sum()} / {len(filled)} ({profitable.mean():.1%})')
    print(f'Total net PnL         : {filled["net_pnl"].sum():.6f}')
    print(f'Mean net PnL          : {filled["net_pnl"].mean():.6f}')

    filled.to_parquet('../data/raw/filled_pnl.parquet', index=False)
    print('Saved data/raw/filled_pnl.parquet')

In [ ]:
# Fill completeness: actual vs intended size
if "intended_size" in filled.columns and "fill_size" in filled.columns:
    filled["fill_completeness_rate"] = filled["fill_size"] / filled["intended_size"]
    partial = filled[filled["fill_completeness_rate"] < 0.8]
    print(f"Partial fills (<80%): {len(partial)} of {len(filled)}")
    print(f"Avg fill completeness: {filled['fill_completeness_rate'].mean():.2%}")
else:
    print("NOTE: intended_size not in schema yet")